# Machine Learning & Introduction to AI Models — Full Practical Course
### Kirsley Chennen @ LGM — INSERM U1112
---
**Duration:** 10 hours  |  **Level:** Beginner → Intermediate  |  **Language:** Python 3

> **🧬 Bioinformatics context:** All algorithms are illustrated with biological or medical datasets
> so that every concept stays meaningful for life-sciences students.

## Course outline
| Session | Topic | Time |
|---------|-------|------|
| TD 1 | Data exploration, preprocessing & classical ML | 30 min |
| TD 2 | Model evaluation, cross-validation & hyper-parameter tuning | 30 min |
| TD 3 | Unsupervised learning — clustering & dimensionality reduction | 30 min |
| TD 4 | Introduction to deep learning (MLP, CNN) | 30 min |
| TD 5 | LLMs, transformers & modern AI — overview & hands-on | 30 min |

## Datasets used
- **Stroke Prediction** (Kaggle) — binary classification on clinical data
- **Breast Cancer Wisconsin** (sklearn) — binary classification
- **Synthetic Ciliopathy Gene Expression** — clustering & dimensionality reduction
- **MNIST digits** — image classification with CNN
- **Ciliopathy protein sequences** — protein LM embeddings (ESM-2)

## 📝 This is the STUDENT version — complete the code cells marked `# YOUR CODE HERE`.


---
# TD 1 — Data Exploration, Preprocessing & Classical Machine Learning (2 h)

## 1.1 What is Machine Learning?

Machine Learning (ML) is a sub-field of Artificial Intelligence that gives computers the ability to
**learn patterns from data** without being explicitly programmed for each situation.

Think of it as the difference between:
- **Traditional programming:** You write rules → Computer applies them
- **Machine Learning:** You provide examples → Computer *learns* the rules

### The three main paradigms

| Paradigm | Core idea | Typical output | Examples |
|----------|-----------|----------------|---------|
| **Supervised** | Learn from labelled examples (input → known output) | Prediction | Classification, Regression |
| **Unsupervised** | Find hidden structure in unlabelled data | Groupings / representations | Clustering, Dimensionality Reduction |
| **Reinforcement** | Learn from reward/penalty signals by trial and error | Policy / strategy | Game playing, robotics |

### The ML pipeline (always the same steps!)
```
Raw data → Explore → Preprocess → Split → Train → Evaluate → (tune) → Deploy
```

> **Why this order matters:** You must split data *before* any preprocessing to avoid **data leakage** —
> accidentally letting information from the test set influence your model training.

### Why is ML relevant in bioinformatics?
- Predict disease status from genomic variants → **classification**
- Group patients by gene-expression profiles → **clustering**
- Predict protein structure properties from sequence → **regression**
- Interpret the effect of variants on splicing → **sequence models**


## 1.2 The Dataset — Stroke Prediction

We will use the **[Stroke Prediction Dataset](https://www.kaggle.com/fedesoriano/stroke-prediction-dataset)**.

According to the WHO, stroke is the 2nd leading cause of death globally (~11 % of all deaths).
Our goal: **predict whether a patient is likely to have a stroke** from clinical features.

### Feature dictionary
| Column | Type | Description |
|--------|------|-------------|
| gender | categorical | Male / Female / Other |
| age | numeric | Patient age (years) |
| hypertension | binary (0/1) | Hypertension diagnosis |
| heart_disease | binary (0/1) | Heart disease diagnosis |
| ever_married | categorical | Yes / No |
| work_type | categorical | children / Govt_job / Never_worked / Private / Self-employed |
| Residence_type | categorical | Urban / Rural |
| avg_glucose_level | numeric | Average blood glucose (mg/dL) |
| bmi | numeric | Body Mass Index |
| smoking_status | categorical | formerly smoked / never smoked / smokes / Unknown |
| **stroke** | **binary (TARGET)** | **1 = had a stroke, 0 = no stroke** |

> 💡 **Key observation before we start:** some columns are numbers, some are text categories.
> ML algorithms only understand numbers, so we must *convert* everything to numeric form.


## Tasks — Data Exploration

1. Import the dataset and set the `id` column as the index
2. Print the shape (rows, columns) and display the first 5 rows
3. Calculate the stroke / no-stroke ratio — is the dataset balanced?
4. Plot histograms of `age` and `avg_glucose_level`, split by stroke status
5. Display the dtype (data type) of each column

## Questions
1. How many patients and features are there?
2. Is the dataset **balanced** or **imbalanced**? Why does this matter for ML?
3. Which features need preprocessing and why?


In [ ]:
import warnings

warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

print("✅ Libraries loaded successfully")


In [ ]:
# 1. Import data ###############################################################
# pd.read_csv() reads a CSV file from disk or URL into a DataFrame
# .set_index() moves a column to become the row labels (index)
df = pd.read_csv("https://www.lbgi.fr/~kchennen/healthcare-dataset-stroke-data.csv")

# Print the shape of the dataframe and the head using df.shape and df.head()
# df.shape returns (n_rows, n_columns)
# YOUR CODE HERE

# df.head() shows the first 5 rows — essential for a quick sanity check
# YOUR CODE HERE


In [ ]:
# 2. Class balance #############################################################
# value_counts(normalize=True) gives proportions instead of raw counts
# YOUR CODE HERE

# Visual: age histogram split by stroke status
# Use pandas built-in dataframe .hist() methods to plot the histogram of the "age" column. Check hist parameters to group by !
# YOUR CODE HERE


In [ ]:
# 3. Column data types ###############################################################
# YOUR CODE HERE


## 1.3 Data Preprocessing

Raw data almost never comes ready to feed into a ML algorithm.
We need to transform it into a consistent **numeric matrix**.

### The three main preprocessing steps

#### 1️⃣ Handle missing values (imputation)
Instead of deleting rows with NaN, we *estimate* missing values:
- **Mean / Median imputation** — replace NaN by column average (simple but ignores correlations)
- **Iterative Imputer (MICE)** — models each column as a function of the others → more accurate

#### 2️⃣ Encode categorical variables
ML cannot work with strings like `"Male"` or `"Urban"`. Two main strategies:
- **Label Encoding:** Male=0, Female=1 → implies an *ordering* that may not exist → use only for ordinal features
- **One-Hot Encoding (OHE):** creates one binary column per category → no spurious ordering

Example for `gender`:
```
gender → gender_Male  gender_Female  gender_Other
Male   →     1             0              0
Female →     0             1              0
```

#### 3️⃣ Scale numerical features
Many algorithms (SVM, KNN, PCA, neural networks) are sensitive to the *magnitude* of features.
A column in range [0, 1] and a column in range [0, 800] would be treated very differently.
- **StandardScaler:** subtract mean, divide by std → zero mean, unit variance (`z-score`)
- **MinMaxScaler:** scale to [0, 1] range

> ⚠️ Tree-based algorithms (Random Forest, XGBoost) are NOT sensitive to scaling.
> But it's generally safe to scale anyway.

![Preprocessing diagram](https://i.imgur.com/mRFk0Sh.png)

## Tasks — Preprocessing
1. Identify and separate columns into categorical, numeric, and binary groups
2. Apply `OneHotEncoder` to categorical columns
3. Apply `IterativeImputer` (handle NaN) then `StandardScaler` to numeric columns
4. Concatenate all parts into a final numpy array `array_data`

## Questions
1. What columns are categorical data, what columns are numeric?
2. What columns are already ready to be used and needs no change?
3. What type of processing do you need to do on categorical data and why?
4. What type of processing do you need to do on numeric data and why?
5. What columns contains missing data ? What type of processing do you need to do in this case?
6. If a categorical column has 5 categories, how many new columns does OHE create?
7. Why use `IterativeImputer` instead of simply filling with the mean?

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Define column groups based on data types and processing needs
# We manually assign each column to a processing strategy
columns_categorical = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status",
]
columns_numeric = ["age", "avg_glucose_level", "bmi"]  # bmi has missing values!
binary_cols = ["hypertension", "heart_disease"]  # already 0/1, no processing needed

print("Categorical (→ One-Hot Encoding):", columns_categorical)
print("Numeric     (→ Impute + Scale)   :", columns_numeric)
print("Binary      (→ keep as-is)       :", binary_cols)


In [ ]:
# ── Step 1: One-Hot Encoding for categorical columns ─────────────────────────
# sparse_output=False → return a dense numpy array (easier to work with)
# handle_unknown="ignore" → if a value appears in test but not train, ignore it
# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Step 2: Impute missing values + Scale numeric columns ─────────────────────
# IterativeImputer trains a regression model for each column with NaN
# predicting its value from all other columns → much better than mean imputation
# YOUR CODE HERE

# StandardScaler: for each feature x → (x - mean) / std
# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Step 3: Assemble final matrix ─────────────────────────────────────────────
# We keep binary columns as-is (already 0/1)
# YOUR CODE HERE

# Target label (what we want to predict)
# YOUR CODE HERE

# np.concatenate stacks arrays side by side (axis=1)
# Order: numeric | one-hot categorical | binary | label
# YOUR CODE HERE

# YOUR CODE HERE


## 1.4 Train / Test Split

We always evaluate a model on data it has **never seen during training**.
This simulates real-world deployment: you train on historical data and predict on new patients.

### The golden rule: split BEFORE fitting any transformer!
If you scale *all* data before splitting, the scaler "sees" the test set values and uses them
to compute the mean/std → this is **data leakage** and gives overly optimistic results.

> ✅ Correct: split first, then fit scaler/imputer on train only, apply to test.
> ❌ Wrong:   fit scaler on all data, then split.

### Why `stratify=Y`?
With an imbalanced target (95% / 5%), a random split might put very few stroke cases
in the test set. `stratify=Y` ensures both splits contain the same proportion of each class.

## Tasks
1. Separate features `X` and label `Y` from `array_data`
2. Split using `train_test_split` with a 70/30 ratio and `stratify=Y`
3. Verify that the stroke ratio is preserved in both splits


In [ ]:
from sklearn.model_selection import train_test_split

# Separate features (all columns except last) and label (last column)
# YOUR CODE HERE

# test_size=0.30 → 30% for testing, 70% for training
# stratify=Y     → preserve class ratio in both splits
# random_state   → fix random seed for reproducibility
# YOUR CODE HERE

# YOUR CODE HERE


## 1.5 Training Your First Classifier

We will start with a **Random Forest** — one of the most reliable all-purpose classifiers.

### What is a Random Forest?
A Random Forest builds many **Decision Trees** (typically 100–500) on random subsets of the data
and random subsets of features, then aggregates their predictions by **majority vote**.

Why is this powerful?
- Individual trees overfit → but their errors are *different* and cancel out when averaged
- Robust to outliers and noisy features
- Gives free **feature importance** estimates
- Works well without much hyperparameter tuning

### The sklearn API (always the same!)
```python
clf = SomeClassifier(hyperparameters...)
clf.fit(X_train, Y_train)          # learn from training data
Y_pred = clf.predict(X_test)       # predict on unseen data
```

## Tasks
1. Train a `RandomForestClassifier` on the training data
2. Compute accuracy on the test set
3. Display the **confusion matrix** and interpret it

## Questions
1. What does accuracy tell you on an imbalanced dataset?
2. Interpret the confusion matrix: which errors are most dangerous medically?

![Confusion Matrix](https://i.imgur.com/EAf5SNh.png)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

# Train the model
# class_weight="balanced" → automatically compensates for class imbalance
# YOUR CODE HERE

# Predict on the test set (the model has NEVER seen these examples)
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# Explanation of the confusion matrix:
# YOUR CODE HERE


## 1.6 Handling Class Imbalance

The stroke dataset is ~95% negative / ~5% positive. This is called **class imbalance**.

**Why is it a problem?**
A model that always predicts "No Stroke" gets 95% accuracy but zero medical utility.
It will completely ignore the minority class (actual strokes).

### Strategies to handle imbalance

| Strategy | How it works | Pros / Cons |
|----------|-------------|-------------|
| **Downsampling** | Remove majority-class samples randomly | Simple, fast / loses data |
| **Oversampling (SMOTE)** | Synthesise new minority-class samples | More data / can overfit |
| **`class_weight="balanced"`** | Weight samples inversely to class frequency | No data change / may not be enough |
| **Adjust decision threshold** | Change the 0.5 cutoff to favor recall | Flexible / needs tuning |

We will use **downsampling** here (simplest to understand).

![Downsampling](https://i.imgur.com/QPOuSKk.png)


In [ ]:
# ── Downsampling: reduce majority class to match minority class ───────────────
# YOUR CODE HERE

# YOUR CODE HERE

# Randomly sample from the majority class to get the same count as minority
# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Re-preprocess the downsampled dataframe ───────────────────────────────────
# IMPORTANT: we use .transform() (not .fit_transform()) here because the
# scaler/imputer were already fitted on the original full training data.
# We must apply the SAME transformation parameters — not refit on new data.
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# Train a new model on the balanced data
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE


---
# TD 2 — Model Evaluation, Cross-validation & Hyperparameter Tuning (2 h)

## 2.1 Beyond Accuracy — Evaluation Metrics

For **imbalanced medical data**, accuracy is misleading.
A model that always predicts "no disease" has 95% accuracy but is clinically useless.

We need metrics that explicitly measure performance on the **minority class** (patients with stroke).

### The confusion matrix vocabulary

```
                  Predicted
                  Negative  Positive
Actual  Negative     TN        FP    ← False Alarm
        Positive     FN        TP    ← Missed Detection
                     ↑
                False Negative = MISSED STROKE → very dangerous!
```

### Key metrics derived from the confusion matrix

| Metric | Formula | Interpretation | Best for |
|--------|---------|----------------|----------|
| **Accuracy** | (TP+TN)/N | Overall correctness | Balanced datasets only |
| **Balanced Accuracy** | (Sensitivity+Specificity)/2 | Fairness across classes | Imbalanced data |
| **Precision** | TP/(TP+FP) | Of predicted positives, how many are real? | When FP is costly |
| **Recall (Sensitivity)** | TP/(TP+FN) | Of real positives, how many did we detect? | **When FN is costly (disease!)** |
| **Specificity** | TN/(TN+FP) | True negative rate | |
| **F1-Score** | 2·P·R/(P+R) | Harmonic mean of Precision & Recall | Imbalanced data |
| **AUC-ROC** | Area under ROC curve | Threshold-independent | General ranking ability |

> 🏥 In medicine, **Recall (Sensitivity)** is often the most important metric.
> A missed stroke (FN) is much worse than a false alarm (FP).

![Metrics reference](https://i.imgur.com/8b5AkS3.png)
![ROC curve](https://i.imgur.com/DFw604d.png)


In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    # YOUR CODE HERE

def get_all_metrics(model, Xte, Yte):
    # YOUR CODE HERE
# Compute metrics for both models
# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── ROC & Precision-Recall curves ─────────────────────────────────────────────
# ROC curve: sensitivity vs (1-specificity) at every possible threshold
# AUC = 1.0 → perfect classifier  |  AUC = 0.5 → random classifier
# YOUR CODE HERE

# YOUR CODE HERE

# Precision-Recall curve: more informative for imbalanced data
# YOUR CODE HERE

# YOUR CODE HERE


## 2.2 Cross-Validation

A single train/test split can give a **biased estimate** of performance depending on
which samples ended up in the test set by chance.

**K-Fold Cross-Validation** solves this by repeating the split K times:

```
Data: [Fold1 | Fold2 | Fold3 | Fold4 | Fold5]

Run 1: Train=[2,3,4,5]  Test=[1]  → Score_1
Run 2: Train=[1,3,4,5]  Test=[2]  → Score_2
...
Run 5: Train=[1,2,3,4]  Test=[5]  → Score_5

Final score = mean(Score_1 … Score_5) ± std
```

**StratifiedKFold** ensures each fold has the same class ratio as the full dataset —
essential for imbalanced data.

> 💡 A large std means your model is **unstable** (high variance) — its performance
> strongly depends on which data it was trained on.

![Cross-validation diagram](https://fr.mathworks.com/discovery/cross-validation/_jcr_content/mainParsys/image.adapt.full.medium.jpg/1623131646985.jpg)


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# StratifiedKFold: K=5 folds, preserving class ratio in each fold
# YOUR CODE HERE

# cross_val_score handles train/test splitting, fitting, and scoring automatically
# n_jobs=-1 → use all CPU cores in parallel (much faster)
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE


## 2.3 Hyperparameter Tuning with Optuna

**Parameters** are values the model *learns from data* (e.g., tree split thresholds).
**Hyperparameters** are settings you choose *before* training (e.g., number of trees, max tree depth).

The goal of hyperparameter tuning is to find the combination that maximises validation performance.

### Tuning strategies

| Method | Strategy | Efficiency |
|--------|----------|-----------|
| **Grid Search** | Try all combinations in a predefined grid | Thorough but exponentially slow |
| **Random Search** | Sample randomly from ranges | Faster, covers more space |
| **Bayesian Optimisation (Optuna)** | Use past results to intelligently pick next trial | ★ Most efficient |

**Optuna** uses Tree-structured Parzen Estimator (TPE) — a Bayesian method that builds a
probabilistic model of the objective function and samples promising regions.

> ⚠️ Always tune on **validation data** (cross-val), never on the test set!
> Using the test set for tuning = data leakage → overoptimistic results.

![Hyperparameter tuning](https://i.imgur.com/3plYcqn.png)


In [ ]:
# Install optuna if needed (uncomment):
# !pip install optuna -q


In [ ]:
import optuna
# YOUR CODE HERE

# YOUR CODE HERE

def objective(trial):
    # YOUR CODE HERE
# Create a study and optimise (maximize F1)
# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Train optimised model and compare ─────────────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE


## 2.4 Feature Importance

Random Forests compute a **feature importance score** as a byproduct of training —
the mean decrease in impurity (Gini index) over all trees for each feature.

This is extremely useful in bioinformatics to:
- Identify the most discriminative **clinical biomarkers**
- Prioritise genes in a multi-omics study
- Remove irrelevant/noisy features to speed up training

> ⚠️ Limitations of this importance measure:
> - Biased toward high-cardinality or continuous features
> - Does not account for feature interactions
> - Alternative: **SHAP values** (SHapley Additive exPlanations) — model-agnostic, more accurate


In [ ]:
# ── Reconstruct feature names from preprocessing steps ────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE


---
# TD 3 — Unsupervised Learning: Clustering & Dimensionality Reduction (2 h)

## 3.1 Introduction to Unsupervised Learning

In **unsupervised learning** there are no labels. The algorithm must discover hidden structure
in the data on its own.

Two main tasks in this session:

### Dimensionality Reduction (DR)
High-dimensional data (e.g., 500 genes per patient) is hard to visualise and analyse.
DR compresses data into 2–3 dimensions while preserving meaningful structure.

| Method | Type | Preserves | Best for |
|--------|------|-----------|----------|
| **PCA** | Linear | Global variance (PC directions) | Preprocessing, batch correction |
| **t-SNE** | Non-linear | Local neighbourhood structure | Visualisation of clusters |
| **UMAP** | Non-linear | Local + some global structure | Single-cell, fast, more stable |

### Clustering
Group samples into clusters based on similarity, without knowing the true labels.

| Method | Requires K? | Principle |
|--------|------------|-----------|
| **K-Means** | Yes | Minimise sum of squared distances to K centroids |
| **Hierarchical** | No | Build a tree by iteratively merging closest clusters |
| **DBSCAN** | No | Density-based: groups by dense regions |

### 🧬 Bioinformatics applications
- PCA of RNA-seq to detect **batch effects** before differential expression
- UMAP of single-cell data to identify **cell populations**
- K-Means / hierarchical clustering of **patient gene-expression profiles** for subtyping
- Identify ciliopathy patient subgroups based on multi-omics data

## Dataset: Synthetic Gene Expression — Ciliopathy Cohort

We simulate a gene expression matrix for 200 patients from 4 ciliopathy disease groups,
each with distinct expression signatures across 500 genes.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

# ── Simulate gene expression data (4 ciliopathy subgroups) ────────────────────
# In reality, you would load your own count matrix here (e.g., from DESeq2 output)
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE


## 3.2 Dimensionality Reduction

### PCA — Principal Component Analysis

PCA finds the **directions of maximum variance** in the data (principal components).
Each PC is a linear combination of all original features.

Mathematical intuition: PCA rotates the data so that the first axis (PC1) captures the most
variance, PC2 captures the second most, and so on — all axes are orthogonal.

### t-SNE
Non-linear method that converts pairwise distances into probabilities and minimises
the KL divergence between high-dimensional and low-dimensional probability distributions.
**Non-deterministic** — different runs give different results (controlled by `random_state`).

### UMAP
Approximates the high-dimensional manifold structure using topological methods.
Generally faster than t-SNE and better at preserving global structure.


In [ ]:
# ── PCA: Linear dimensionality reduction ──────────────────────────────────────
# n_components=2: project to 2D for visualisation
# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── t-SNE: Non-linear projection ──────────────────────────────────────────────
# perplexity: roughly = number of nearest neighbours considered (typical: 5-50)
# n_iter: more iterations → better convergence (but slower)
# YOUR CODE HERE

# ── Side-by-side comparison ───────────────────────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── UMAP ─────────────────────────────────────────────────────────────────────
# Install: pip install umap-learn
# YOUR CODE HERE
    import umap
    # YOUR CODE HERE

    # YOUR CODE HERE


## 3.3 Clustering

### How to choose K for K-Means?

Two complementary approaches:
- **Elbow method:** plot inertia (sum of squared distances to centroid) vs K.
  Find the "elbow" point where inertia stops decreasing steeply.
- **Silhouette score:** measures cohesion (how similar a point is to its cluster) vs
  separation (how different from other clusters). Range: −1 to 1; higher is better.

### Evaluation with known labels: Adjusted Rand Index (ARI)
When we know the true labels (here: disease groups), we can measure clustering quality:
- **ARI = 1.0** → perfect match between clusters and true labels
- **ARI = 0.0** → no better than random assignment


In [ ]:
# ── Elbow curve: find optimal K ───────────────────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── K-Means with K=4 (matching the true number of disease groups) ─────────────
# YOUR CODE HERE

# YOUR CODE HERE

# ── Agglomerative clustering ───────────────────────────────────────────────────
# YOUR CODE HERE

# ── Visualise K-Means clusters vs true labels ──────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE


---
# TD 4 — Introduction to Deep Learning (2 h)

## 4.1 From Classical ML to Neural Networks

| Aspect | Classical ML | Deep Learning |
|--------|-------------|----------------|
| Features | Manual engineering required | Learned automatically from raw data |
| Data needs | Works on small datasets | Needs large amounts of data |
| Training speed | Fast (seconds–minutes) | Slow (minutes–hours, GPU helps a lot) |
| Interpretability | Generally interpretable | Often "black box" |
| Libraries | scikit-learn | PyTorch, TensorFlow, Keras |
| Typical tasks | Tabular data | Images, sequences, text, audio |

### The artificial neuron (perceptron)

A single neuron computes a **weighted sum** of its inputs and applies an **activation function**:

```
output = activation( w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ + bias )
```

Common activation functions:
- **ReLU(x) = max(0, x)** — most used in hidden layers (fast, avoids vanishing gradients)
- **Sigmoid(x) = 1/(1+e⁻ˣ)** — squashes to [0,1], used for binary output
- **Softmax** — squashes to sum=1 probability distribution, used for multi-class output

### Multi-Layer Perceptron (MLP) — "Fully Connected Network"

```
Input layer  →  Hidden layer 1  →  Hidden layer 2  →  Output layer
[x₁…xₙ]         [ReLU neurons]     [ReLU neurons]     [Sigmoid/Softmax]
```

### Key training concepts
- **Loss function:** measures how wrong the predictions are (we minimise this)
  - Binary classification: Binary Cross-Entropy (BCE)
  - Multi-class: Categorical Cross-Entropy
  - Regression: Mean Squared Error (MSE)
- **Backpropagation:** computes gradients of loss w.r.t. all weights via chain rule
- **Optimiser (Adam):** updates weights using gradients (smarter version of SGD)
- **Epoch:** one full pass over the training data
- **Batch size:** number of samples per gradient update
- **Dropout:** randomly sets neurons to zero during training → prevents overfitting

## 4.2 Dataset: Breast Cancer Wisconsin

Binary classification: malignant (1) vs benign (0).
30 numerical features from digitized FNA (Fine Needle Aspiration) cell nuclei images.


In [ ]:
# Install PyTorch if needed (uncomment):
# !pip install torch torchvision -q


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler as SS2
from sklearn.model_selection import train_test_split as tts2
from sklearn.metrics import classification_report

# ── Load & prepare Breast Cancer data ─────────────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE

# Scale features (important for neural networks!)
# YOUR CODE HERE

# YOUR CODE HERE

# Convert numpy arrays to PyTorch tensors (required by PyTorch)
# YOUR CODE HERE

# DataLoader batches the data for mini-batch gradient descent
# YOUR CODE HERE


In [ ]:
# ── Define MLP architecture ───────────────────────────────────────────────────
class MLP(nn.Module):
    # YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
# Each EPOCH = one full pass through all training data
# YOUR CODE HERE

# YOUR CODE HERE

    # YOUR CODE HERE

    # YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE

from sklearn.metrics import ConfusionMatrixDisplay
# YOUR CODE HERE


## 4.3 Convolutional Neural Networks (CNN)

CNNs are designed for **structured / spatial data** — images, but also DNA/protein sequences!

### Key CNN building blocks

| Layer | Purpose | Parameters |
|-------|---------|-----------|
| **Conv2d** | Apply learnable filters to detect local patterns | kernel_size, n_filters |
| **ReLU** | Non-linear activation after each conv | — |
| **MaxPool2d** | Reduce spatial dimensions by taking max in window | pool_size |
| **Flatten** | Convert 2D feature maps to 1D vector | — |
| **Linear (FC)** | Final classification | — |

### Why CNNs for images (and not MLP)?
- **Parameter sharing:** the same filter is applied at every position → fewer parameters
- **Translation invariance:** the cat is a cat whether it's in the top-left or bottom-right
- **Hierarchical features:** shallow layers detect edges, deep layers detect complex shapes

### Bioinformatics applications of CNNs
- Predict **enhancer / promoter / splice site** from DNA sequence (1D conv)
- Analyse **histology images** (H&E slides) for cancer detection (2D conv)
- Predict **protein contact maps** from MSA (2D conv)

## Dataset: MNIST (28×28 grayscale handwritten digit images, 10 classes)


In [ ]:
import torchvision
import torchvision.transforms as transforms

# ── Load MNIST ────────────────────────────────────────────────────────────────
# Transforms: convert to tensor, then normalise (mean=0.1307, std=0.3081 for MNIST)
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE

# Visualise a few samples
# YOUR CODE HERE


In [ ]:
# ── Define CNN architecture ───────────────────────────────────────────────────
class SimpleCNN(nn.Module):
    # YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Train CNN ─────────────────────────────────────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Evaluate CNN on test set ──────────────────────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE

# ── Visualise predictions ─────────────────────────────────────────────────────
# YOUR CODE HERE

# YOUR CODE HERE


---
# TD 5 — Large Language Models, Transformers & Modern AI (2 h)

## 5.1 The Transformer Architecture

The **Transformer** (Vaswani et al., "Attention is All You Need", 2017) is the foundation
of virtually all modern AI: ChatGPT, AlphaFold, ESM-2, DALL-E, etc.

### The key innovation: Self-Attention

Before Transformers, sequences were processed step-by-step (RNNs/LSTMs).
Self-attention allows the model to look at **all tokens simultaneously** and learn which
tokens are relevant to each other for the task at hand.

```
Attention(Q, K, V) = softmax(Q·Kᵀ / √dₖ) · V

Q = Queries (what am I looking for?)
K = Keys    (what do I contain?)
V = Values  (what do I return?)
```

### Multi-Head Attention
Run attention multiple times in parallel with different learned projections →
the model can attend to different aspects of the sequence simultaneously.

### Major model families

| Model | Architecture | Training objective | Key use case |
|-------|-------------|-------------------|--------------|
| **BERT** | Encoder only | Masked language modelling | Classification, NER |
| **GPT** | Decoder only | Next token prediction | Text generation |
| **T5** | Encoder-Decoder | Text-to-text tasks | Translation, Q&A |
| **ESM-2** | Encoder (protein) | Masked amino acid prediction | Protein embeddings |
| **Nucleotide Transformer** | Encoder (DNA) | Masked nucleotide prediction | Genomic variant effect |
| **BioGPT** | Decoder (biomedical) | Next token on PubMed | Literature mining |

## 5.2 Transformers in Bioinformatics

Protein language models (pLMs) treat amino acid sequences as "sentences":

- **ESM-2** (Meta AI, 2022): 8M–15B parameters trained on 250M UniProt sequences.
  Captures evolutionary information and structural properties from sequence alone.
- **AlphaFold2** uses attention internally to predict 3D structure from sequence.
- **Nucleotide Transformer** (InstaDeepAI, 2023): trained on 850 genomes,
  state-of-the-art for regulatory element prediction and variant effect scoring.

### Embeddings
Instead of raw sequences, these models produce **dense vector representations** (embeddings).
Similar sequences/proteins have similar embeddings — we can use PCA/UMAP to visualise
evolutionary or functional relationships!


In [ ]:
# Install if needed:
# !pip install transformers sentencepiece -q


In [ ]:
# ── Load ESM-2 (8M parameter protein language model from Meta AI) ──────────────
# ESM-2 is trained on millions of protein sequences and learns rich representations
# YOUR CODE HERE
    from transformers import EsmTokenizer, EsmModel
    import torch

    # YOUR CODE HERE

    # Representative sequences (N-terminal region) from ciliopathy proteins
    # YOUR CODE HERE

    # Tokenise each sequence and extract mean-pooled embedding
    # YOUR CODE HERE
            # Tokeniser converts amino acid sequence to integer IDs
            # YOUR CODE HERE
            # last_hidden_state: [1, seq_len, 320]
            # Mean pool over sequence (excluding CLS and EOS special tokens)
            # YOUR CODE HERE

    # YOUR CODE HERE

# YOUR CODE HERE


In [ ]:
# ── Visualise protein embeddings with PCA ─────────────────────────────────────
# Even with only 6 proteins, PCA shows which proteins have similar sequence features
# YOUR CODE HERE

# YOUR CODE HERE

# YOUR CODE HERE


## 5.3 Zero-Shot and Few-Shot Learning with Text LLMs

Modern LLMs can solve tasks they were **never explicitly fine-tuned on** by understanding
natural language instructions in the prompt.

| Paradigm | Description | Example |
|----------|-------------|---------|
| **Zero-shot** | Model receives only the instruction, no examples | "Classify this text as positive or negative" |
| **Few-shot** | Model receives the instruction + a few labelled examples | "Here are 3 examples, now classify this one" |
| **Fine-tuning** | Supervised training on task-specific labelled dataset | Train on 10,000 labelled abstracts |
| **RAG** | Retrieval-Augmented Generation — inject relevant docs as context | "Using this paper, answer: ..." |

### Prompt engineering
The art of crafting prompts that guide the LLM to produce the desired output.
Key principles: be specific, give context, specify format, use examples.

## 5.4 Zero-Shot Text Classification for Biomedical Abstracts

We use a pre-trained **BART-large-MNLI** model as a zero-shot classifier.
It was trained on Natural Language Inference (NLI) tasks and can classify text into
arbitrary categories without any task-specific fine-tuning.


In [ ]:
from transformers import pipeline

# ── Zero-shot classification ───────────────────────────────────────────────────
# This model was never explicitly trained to classify scientific abstracts,
# yet it can do it by framing classification as entailment
# YOUR CODE HERE

# Our candidate labels (the categories we want to classify into)
# YOUR CODE HERE

# Three test abstracts from different fields
# YOUR CODE HERE

# YOUR CODE HERE


## 5.5 Reflection: LLMs for Rare Disease Research

### What can biomedical LLMs do?

| Application | Model | Approach |
|-------------|-------|----------|
| Variant effect prediction | ESM-2, Nucleotide Transformer | Zero-shot scoring of mutant vs wild-type |
| Literature mining | BioGPT, PubMedBERT | NER, relation extraction |
| Gene ontology prediction | ProtT5, ESM-2 | Embedding + classifier |
| Protein function annotation | ESM-2 | Embedding similarity to annotated proteins |
| Clinical note analysis | BioBERT, GatorTron | Zero-shot or fine-tuned classification |
| Drug-target interaction | MolBERT | Molecular fingerprint similarity |

### Limitations to keep in mind
- LLMs can **hallucinate** (confidently produce incorrect information)
- Performance degrades on very rare entities (low frequency in training data)
- Opaque reasoning — hard to verify *why* a prediction was made
- High computational cost for large models
- Training data has a knowledge cutoff date

### 🧬 Discussion question
> Think of one specific task in your ciliopathy research where a protein LM (ESM-2)
> or a biomedical text LM (BioGPT) could accelerate your work.
> What would be the input, the output, and the validation strategy?


---
# Summary & Cheatsheet

## Full ML Pipeline

```
1.  Explore data          → df.describe(), .hist(), .isnull()
2.  Preprocess            → OHE, StandardScaler, IterativeImputer
3.  Split data            → train_test_split(stratify=y)   ← BEFORE fitting transformers!
4.  Train model           → clf.fit(X_train, y_train)
5.  Evaluate              → accuracy, F1, AUC-ROC, confusion matrix
6.  Handle imbalance      → downsampling, SMOTE, class_weight
7.  Cross-validate        → StratifiedKFold + cross_val_score
8.  Tune hyperparameters  → Optuna Bayesian optimisation
9.  Explain model         → feature_importances_, SHAP values
10. Cluster / reduce dim  → PCA → t-SNE/UMAP → K-Means
11. Deep learning         → PyTorch MLP / CNN → train loop + eval
12. LLM embeddings        → HuggingFace transformers → ESM-2, BERT
```

## Algorithm selection guide

| Data type | Size | Interpretability | Recommended algorithm |
|-----------|------|------------------|-----------------------|
| Tabular | Small | High | Logistic Regression, Decision Tree |
| Tabular | Medium | Medium | Random Forest, SVM |
| Tabular | Large | Low | XGBoost, LightGBM |
| Images | Any | Low | CNN (ResNet, EfficientNet) |
| Text / sequences | Any | Low | Transformer (BERT, ESM-2) |
| Unlabelled | Any | — | PCA → UMAP → K-Means |

## Resources

| Resource | URL |
|----------|-----|
| Google ML Crash Course | https://developers.google.com/machine-learning/crash-course |
| Scikit-learn user guide | https://scikit-learn.org/stable/user_guide.html |
| HuggingFace NLP course | https://huggingface.co/learn/nlp-course |
| ESM-2 protein LM | https://github.com/facebookresearch/esm |
| Nucleotide Transformer | https://huggingface.co/InstaDeepAI |
| Optuna documentation | https://optuna.readthedocs.io |
| AI Roadmap | https://i.am.ai/roadmap |
| Python visualisation gallery | https://www.python-graph-gallery.com |
